# ArtLab Backend v2.0 — Colab Notebook

Servidor FastAPI con SDXL 1.0 + Qwen GGUF para el dashboard.

## Setup inicial
1. Runtime → **Cambiar tipo de entorno de ejecución** → **T4 GPU**
2. Conecta Google Drive (opcional, para modelos locales):
   `from google.colab import drive; drive.mount('/content/drive')`
3. Ejecuta todo en orden (Ctrl+F9 o Runtime → Ejecutar todo)

---


In [ ]:
# @title 1. Instalar dependencias
!pip install -q fastapi uvicorn[standard] websockets pydantic psutil pyngrok nest_asyncio
!pip install -q torch diffusers transformers accelerate sentencepiece
# llama-cpp-python: intentar rueda pre-compilada (segundos), si no existe compilar (~15 min)
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 2>/dev/null
# Verificar que quedó instalado; si no, compilar desde fuente
!python -c 'import llama_cpp; print("✅ llama-cpp-python instalado")' 2>/dev/null || (CMAKE_ARGS='-DLLAMA_CUDA=on' pip install -v llama-cpp-python 2>&1 | tail -20)
# xformers es opcional
!pip install -q xformers 2>/dev/null; echo '✅ Dependencias instaladas'

---
## 2. Imports y configuración
---


In [ ]:
import os, json, time, asyncio, threading, logging, gc
from datetime import datetime
from pathlib import Path
from typing import Optional, Any

import psutil
import torch
from fastapi import FastAPI, HTTPException, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger('artlab')

# ── Rutas ──
DATA_DIR = Path('/content/artlab_data')
MODELS_DIR = Path('/content/artlab_models')
DRIVE_MODELS = Path('/content/drive/MyDrive/artlab/models')
for p in [DATA_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('✅ Imports listos')

---
## 3. VRAM Manager
---

La T4 tiene 16 GB VRAM. SDXL ocupa ~8-10 GB, Qwen 9B Q4 ~5-6 GB.
No pueden convivir. El **swap solo ocurre cuando el usuario lo fuerza**
(`?force=true` al cargar un modelo, o al intentar chatear/generar).
No hay swap automático al cargar.


In [ ]:
class VRAMManager:
    """Solo trackea el modelo activo. Swap solo con force=True explícito."""

    def __init__(self):
        self.active_model: str | None = None

    def set_active(self, model_type: str):
        self.active_model = model_type

    def clear_active(self):
        self.active_model = None

    def unload_all(self):
        unload_sdxl()
        unload_qwen()
        self.active_model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        log.info('🧹 VRAM liberada')


def conflict_msg(model_id: str) -> str | None:
    """Si hay conflicto de VRAM, devuelve un mensaje."""
    if model_id == 'sdxl' and QWEN_LLM is not None:
        return 'Qwen está ocupando VRAM. Usa ?force=true para descargarlo y cargar SDXL.'
    if model_id == 'qwen' and SDXL_PIPELINE is not None:
        return 'SDXL está ocupando VRAM. Usa ?force=true para descargarlo y cargar Qwen.'
    return None


vram = VRAMManager()
print('✅ VRAM Manager listo')

---
## 4. SDXL Pipeline
---

Carga el modelo custom **`BalancedAllInOnePhotorealAnime_v10.safetensors`** (CivitAI).

⚠️ **Importante:**
- NO usa `from_pretrained()` — NO descarga nada de HuggingFace
- Usa `from_single_file()` — carga directo desde **Google Drive**
- Es un merge "vae-baked" (todo incluido), no necesita modelo base
- El modelo debe estar en Drive. Si no está, el servidor no arranca SDXL


In [ ]:
SDXL_PIPELINE = None
SDXL_MODEL_PATH: str | None = None
MODEL_FILENAME = 'BalancedAllInOnePhotorealAnime_v10.safetensors'

# @markdown ---
# @markdown ### Configuración
# @title Ruta exacta del safetensor en Google Drive
SDXL_DRIVE_PATH = '/content/drive/MyDrive/artlab/models/BalancedAllInOnePhotorealAnime_v10.safetensors'  # @param {type:'string'}


def find_sdxl_model() -> str:
    p = Path(SDXL_DRIVE_PATH)
    if p.exists():
        log.info(f'📂 SDXL encontrado en: {p}')
        return str(p)
    raise FileNotFoundError(
        f'❌ No se encontró el modelo en:\n  {p}\n\n'
        '1. Monta Google Drive: from google.colab import drive; drive.mount(\'/content/drive\')\n'
        '2. Sube el safetensor a esa ruta o cambia SDXL_DRIVE_PATH arriba\n'
        '3. Si no tienes el modelo, descárgalo desde: https://civitai.com/models/...'
    )


async def load_sdxl():
    global SDXL_PIPELINE, SDXL_MODEL_PATH
    if SDXL_PIPELINE is not None:
        return {'ok': True, 'msg': 'SDXL ya cargado'}
    try:
        model_path = find_sdxl_model()
        log.info('🔄 Cargando SDXL (esto toma ~30s)...')
        from diffusers import StableDiffusionXLPipeline
        SDXL_PIPELINE = StableDiffusionXLPipeline.from_single_file(
            model_path, torch_dtype=torch.float16, use_safetensors=True,
        )
        # enable_model_cpu_offload reemplaza a to('cuda') — óptimo para T4 16GB
        SDXL_PIPELINE.enable_model_cpu_offload()
        SDXL_PIPELINE.enable_vae_slicing()
        SDXL_PIPELINE.enable_vae_tiling()
        if hasattr(SDXL_PIPELINE, 'enable_xformers_memory_efficient_attention'):
            SDXL_PIPELINE.enable_xformers_memory_efficient_attention()
        SDXL_MODEL_PATH = model_path
        vram.set_active('sdxl')
        log.info('✅ SDXL cargado y optimizado')
        return {'ok': True, 'msg': 'SDXL cargado correctamente'}
    except Exception as e:
        SDXL_PIPELINE = None
        log.error(f'❌ Error cargando SDXL: {e}')
        raise HTTPException(500, str(e))


def unload_sdxl():
    global SDXL_PIPELINE, SDXL_MODEL_PATH
    if SDXL_PIPELINE is not None:
        del SDXL_PIPELINE
        SDXL_PIPELINE = None
        SDXL_MODEL_PATH = None
        if vram.active_model == 'sdxl':
            vram.clear_active()
        log.info('⏏️ SDXL descargado de VRAM')


async def generate_image(prompt: str, negative_prompt: str = '',
                         steps: int = 20, cfg: float = 7,
                         seed: int | None = None,
                         width: int = 1024, height: int = 1024):
    if SDXL_PIPELINE is None:
        raise HTTPException(400, 'SDXL no está cargado. Cárgalo desde Settings → Sistema.')
    generator = None
    if seed is not None:
        generator = torch.Generator('cuda').manual_seed(seed)
    log.info(f'🎨 Generando: steps={steps}, cfg={cfg}, seed={seed}')
    result = SDXL_PIPELINE(
        prompt=prompt, negative_prompt=negative_prompt or None,
        num_inference_steps=steps, guidance_scale=cfg,
        generator=generator, width=width, height=height,
    )
    return result.images[0]


print('✅ SDXL config listo')

---
## 5. Qwen GGUF Pipeline
---

Carga Qwen desde HuggingFace en formato GGUF.
El usuario puede seleccionar el nivel de cuantización.

**Modelo:** `HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive`
NSFW-ready, 262K contexto, visión multimodal.

Quants disponibles:
- `Q4_K_M` (5.3 GB) ← recomendado T4
- `Q6_K` (6.9 GB)
- `Q8_0` (8.9 GB)
- `BF16` (17 GB) — no cabe en T4


In [ ]:
QWEN_LLM = None
QWEN_MODEL_PATH: str | None = None
MMPROJ_PATH: str | None = None
QWEN_KWARGS: dict = {}

# @markdown ---
# @markdown ### Configuración del LLM

# @title Repo en HuggingFace
QWEN_HF_REPO = 'HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive'  # @param {type:'string'}

# @title Quantización (BF16 no cabe en T4)
QWEN_QUANT = 'Q4_K_M'  # @param ['Q4_K_M','Q6_K','Q8_0']
QWEN_CTX = 131072  # @param {type:'integer'} — mínimo 128K para thinking
QWEN_GPU_LAYERS = -1  # @param {type:'integer'}

# @title Modo thinking (más razonamiento vs respuestas directas)
QWEN_THINKING = False  # @param {type:'boolean'}

QWEN_HF_FILENAME = f'Qwen3.5-9B-Uncensored-HauhauCS-Aggressive-{QWEN_QUANT}.gguf'
MMPROJ_FILENAME = 'mmproj-Qwen3.5-9B-Uncensored-HauhauCS-Aggressive-BF16.gguf'


async def load_qwen():
    global QWEN_LLM, QWEN_MODEL_PATH, MMPROJ_PATH, QWEN_KWARGS
    if QWEN_LLM is not None:
        return {'ok': True, 'msg': 'Qwen ya cargado'}
    try:
        from huggingface_hub import hf_hub_download
        from llama_cpp import Llama
        log.info(f'⬇️  Descargando Qwen GGUF ({QWEN_QUANT}) desde {QWEN_HF_REPO}...')
        model_path = hf_hub_download(
            repo_id=QWEN_HF_REPO, filename=QWEN_HF_FILENAME,
            repo_type='model', resume_download=True,
        )
        try:
            MMPROJ_PATH = hf_hub_download(
                repo_id=QWEN_HF_REPO, filename=MMPROJ_FILENAME,
                repo_type='model', resume_download=True,
            )
        except Exception:
            MMPROJ_PATH = None
            log.warning('⚠️  mmproj no encontrado — sin visión')
        log.info('🔄 Cargando Qwen en VRAM...')
        mmproj_kw = {'mmproj': MMPROJ_PATH} if MMPROJ_PATH else {}
        QWEN_LLM = Llama(
            model_path=model_path, n_ctx=QWEN_CTX,
            n_gpu_layers=QWEN_GPU_LAYERS, n_threads=2,
            verbose=False, flash_attn=True, **mmproj_kw,
        )
        QWEN_MODEL_PATH = model_path
        QWEN_KWARGS = {
            'max_tokens': 4096 if QWEN_THINKING else 2048,
            'temperature': 0.6 if QWEN_THINKING else 0.7,
            'top_p': 0.95 if QWEN_THINKING else 0.8,
            'top_k': 20, 'repeat_penalty': 1.0,
        }
        vram.set_active('qwen')
        log.info('✅ Qwen cargado')
        return {'ok': True, 'msg': f'Qwen3.5-9B {QWEN_QUANT} cargado'}
    except Exception as e:
        QWEN_LLM = None
        log.error(f'❌ Error cargando Qwen: {e}')
        raise HTTPException(500, str(e))


def unload_qwen():
    global QWEN_LLM, QWEN_MODEL_PATH
    if QWEN_LLM is not None:
        del QWEN_LLM
        QWEN_LLM = None
        QWEN_MODEL_PATH = None
        if vram.active_model == 'qwen':
            vram.clear_active()
        log.info('⏏️ Qwen descargado de VRAM')


async def chat_qwen(messages: list[dict]) -> str:
    if QWEN_LLM is None:
        raise HTTPException(400, 'Qwen no está cargado. Cárgalo desde Settings → Sistema.')
    result = QWEN_LLM.create_chat_completion(messages=messages, **QWEN_KWARGS)
    return result['choices'][0]['message']['content']


print('✅ Qwen config listo')

---
## 6. Persistencia
---


In [ ]:
DATA_FILE = DATA_DIR / 'state.json'
PREFS_FILE = DATA_DIR / 'preferences.json'
SYNC_FILE = DATA_DIR / 'sync.json'

def load_json(path: Path, default: Any = None):
    if path.exists():
        try: return json.loads(path.read_text())
        except: pass
    return {}

def save_json(path: Path, data: Any):
    path.write_text(json.dumps(data, indent=2, default=str))

def get_sync_data() -> dict:
    return load_json(SYNC_FILE, {'data': None, 'timestamp': 0})

def set_sync_data(data: str):
    save_json(SYNC_FILE, {'data': data, 'timestamp': time.time()})

def get_preferences() -> dict:
    return load_json(PREFS_FILE, {})

def set_preferences(data: dict):
    save_json(PREFS_FILE, data)

connected_ws: list[WebSocket] = []

async def broadcast(message: str):
    for ws in connected_ws[:]:
        try: await ws.send_text(message)
        except: connected_ws.remove(ws)

print('✅ Persistencia lista')

---
## 7. FastAPI — Endpoints
---


In [ ]:
app = FastAPI(title='ArtLab Backend', version='2.0.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class ChatRequest(BaseModel):
    message: str
    context: Optional[str] = None
    image: Optional[str] = None
    history: Optional[list] = None

class ChatResponse(BaseModel):
    reply: str
    reasoning: Optional[str] = None
    images: Optional[list[str]] = None
    messageId: str

class GenerateRequest(BaseModel):
    prompt: str
    negativePrompt: Optional[str] = ''
    seed: Optional[int] = None
    cfg: Optional[float] = 7.0
    steps: Optional[int] = 20
    pipeline: Optional[str] = 't2i'
    ti2iStrength: Optional[float] = 0.8
    referenceImage: Optional[str] = None
    width: Optional[int] = 1024
    height: Optional[int] = 1024

class GenerateResponse(BaseModel):
    image: str
    seed: int
    info: Optional[dict] = None

class SyncPushRequest(BaseModel):
    data: str
    timestamp: int

class PreferencesRequest(BaseModel):
    data: str

print('✅ FastAPI configurado')

In [ ]:
# ── Health ──
@app.get('/health')
async def health():
    return {
        'status': 'online',
        'model_loaded': SDXL_PIPELINE is not None or QWEN_LLM is not None,
        'sdxl_loaded': SDXL_PIPELINE is not None,
        'qwen_loaded': QWEN_LLM is not None,
        'active_model': vram.active_model,
        'uptime': time.time(),
    }

# ── System ──
@app.get('/system')
async def system():
    ram = psutil.virtual_memory()
    disk = psutil.disk_usage('/')
    vram_free = vram_total = 0
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram_total = props.total_mem / 1e9
        vram_free = vram_total - torch.cuda.memory_allocated(0) / 1e9
    return {
        'ram': {'total': round(ram.total / 1e9, 1), 'used': round(ram.used / 1e9, 1)},
        'vram': {'total': round(vram_total, 1), 'used': round(vram_total - vram_free, 1)},
        'disk': {'total': round(disk.total / 1e9, 1), 'used': round(disk.used / 1e9, 1)},
    }

# ── Models ──
MODEL_REGISTRY = [
    {'id': 'sdxl', 'name': 'SDXL (BalancedAllInOnePhotorealAnime_v10)', 'type': 'sdxl', 'size': 6.46},
    {'id': 'qwen', 'name': f'Qwen3.5-9B {QWEN_QUANT} (Uncensored)', 'type': 'llm', 'size': 0},
]

@app.get('/models')
async def list_models():
    statuses = {
        'sdxl': 'loaded' if SDXL_PIPELINE else 'unloaded',
        'qwen': 'loaded' if QWEN_LLM else 'unloaded',
    }
    return [
        {**m, 'status': statuses.get(m['id'], 'unloaded'),
         'path': SDXL_MODEL_PATH if m['id'] == 'sdxl' else QWEN_MODEL_PATH}
        for m in MODEL_REGISTRY
    ]

@app.post('/models/{model_id}/load')
async def api_load_model(model_id: str, force: bool = False):
    if model_id not in ('sdxl', 'qwen'):
        raise HTTPException(404, f'Modelo {model_id} no encontrado')
    msg = conflict_msg(model_id)
    if msg:
        if force:
            vram.unload_all()
        else:
            raise HTTPException(409, msg)
    if model_id == 'sdxl':
        return await load_sdxl()
    return await load_qwen()

@app.post('/models/{model_id}/unload')
async def api_unload_model(model_id: str):
    if model_id == 'sdxl':
        unload_sdxl()
    elif model_id == 'qwen':
        unload_qwen()
    else:
        raise HTTPException(404, f'Modelo {model_id} no encontrado')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'ok': True, 'msg': f'{model_id} descargado'}

# ── Chat (Qwen) ──
@app.post('/chat', response_model=ChatResponse)
async def chat(req: ChatRequest):
    import uuid
    if QWEN_LLM is None:
        return ChatResponse(
            reply='⚠️ Qwen no está cargado. Ve a Settings → Sistema y haz clic en Load.',
            reasoning=None,
            messageId=f'noop-{uuid.uuid4().hex[:8]}',
        )
    messages = []
    if req.history:
        for h in req.history:
            role = 'assistant' if h.get('role') == 'assistant' else 'user'
            content = h.get('content', '')
            if h.get('images'):
                content += ' [imagen adjunta]'
            messages.append({'role': role, 'content': content})
    user_msg = req.message
    if req.context:
        user_msg = f'[Contexto]:\n{req.context}\n\n{user_msg}'
    if req.image:
        user_msg += '\n[imagen adjunta en base64]'
    messages.append({'role': 'user', 'content': user_msg})
    try:
        reply = await chat_qwen(messages)
        return ChatResponse(reply=reply, messageId=f'qwen-{uuid.uuid4().hex[:8]}')
    except Exception as e:
        log.error(f'Error en chat Qwen: {e}')
        return ChatResponse(reply=f'Error: {e}', messageId=f'err-{uuid.uuid4().hex[:8]}')

# ── Generate (SDXL) ──
@app.post('/generate', response_model=GenerateResponse)
async def generate(req: GenerateRequest):
    import base64, io, uuid
    from PIL import Image
    if SDXL_PIPELINE is None:
        raise HTTPException(400, 'SDXL no está cargado. Cárgalo desde Settings → Sistema.')
    seed = req.seed or int(time.time()) % (2**32)
    image = await generate_image(
        prompt=req.prompt, negative_prompt=req.negativePrompt or '',
        steps=req.steps, cfg=req.cfg, seed=seed,
        width=req.width, height=req.height,
    )
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    return GenerateResponse(
        image=b64, seed=seed,
        info={'pipeline': req.pipeline, 'steps': req.steps, 'cfg': req.cfg},
    )

# ── Build Prompt ──
@app.post('/build_prompt')
async def build_prompt(req: dict):
    return {'prompt': json.dumps(req, indent=2)}

# ── Sync ──
@app.post('/sync/push')
async def sync_push(req: SyncPushRequest):
    set_sync_data(req.data)
    await broadcast(req.data)
    return {'ok': True}

@app.get('/sync/pull')
async def sync_pull():
    return get_sync_data()

@app.post('/sync/preferences')
async def sync_preferences(req: PreferencesRequest):
    try:
        parsed = json.loads(req.data)
        set_preferences(parsed)
        await broadcast(json.dumps({'type': 'preferences', 'data': parsed}))
        return {'ok': True}
    except json.JSONDecodeError as e:
        raise HTTPException(400, f'Invalid JSON: {e}')

@app.websocket('/sync/ws')
async def sync_websocket(websocket: WebSocket):
    await websocket.accept()
    connected_ws.append(websocket)
    log.info('🟢 WebSocket conectado')
    try:
        while True:
            data = await websocket.receive_text()
            set_sync_data(data)
            await broadcast(data)
    except WebSocketDisconnect:
        if websocket in connected_ws:
            connected_ws.remove(websocket)
        log.info('🔴 WebSocket desconectado')

print('✅ Endpoints registrados')

---
## 8. ngrok — Tunnel público
---


In [ ]:
# @title 8. Iniciar ngrok
from pyngrok import ngrok, conf
from google.colab import userdata

# @markdown Necesitas un token de ngrok. Consíguelo en:
# @markdown https://dashboard.ngrok.com/get-started/your-authtoken

# @title Token de ngrok
NGROK_TOKEN = ''  # @param {type:'string'}

# Intentar desde secrets de Colab > desde form input
if not NGROK_TOKEN:
    try:
        NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    except Exception:
        pass

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    conf.get_default().region = 'us'
    tunnel = ngrok.connect(8000, bind_tls=True)
    public_url = tunnel.public_url.replace('https://', 'https://')
    print(f"{'='*60}")
    print(f'  🚀 ArtLab Backend URL:')
    print(f'  {public_url}')
    print(f"{'='*60}")
    print()
    print('  Pega esta URL en el dashboard → Settings → Conexión')
    print(f'  Docs: {public_url}/docs')
else:
    print('⚠️  No hay token de ngrok.')
    print('  Para obtener URL pública:')
    print('  1. Crea cuenta gratis en https://dashboard.ngrok.com')
    print('  2. Copia tu token de https://dashboard.ngrok.com/get-started/your-authtoken')
    print('  3. Pégalo arriba o guárdalo en Colab Secrets como NGROK_TOKEN')

---
## 9. Iniciar servidor
---


In [ ]:
# @title 9. Iniciar FastAPI
import nest_asyncio
import uvicorn
from IPython.display import clear_output

nest_asyncio.apply()

try:
    base_url = public_url
    token_set = bool(NGROK_TOKEN)
except NameError:
    base_url = 'http://localhost:8000'
    token_set = False

clear_output(wait=True)
print(f"{'='*60}")
print(f'  🎨 ArtLab Backend corriendo')
if token_set:
    print(f'  🌐 URL pública: {base_url}')
    print(f'  💚 Health: {base_url}/health')
else:
    print(f'  💻 Local: {base_url}')
    print('  ⚠️  Sin ngrok — solo accesible desde este Colab')
print(f"{'='*60}")

uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

---
## 📖 Flujo de trabajo

1. **Monta Drive** → sube `BalancedAllInOnePhotorealAnime_v10.safetensors` a `artlab/models/`
2. Ejecuta todo (Runtime → Run all)
3. Copia la URL de ngrok de la celda 8
4. Dashboard → Settings → Conexión → pega URL → Test
5. **Carga un modelo:** Settings → Sistema → Load
   - SDXL → carga directo desde Drive (`from_single_file`, 0 descargas HF)
   - Qwen → se descarga automáticamente desde HuggingFace (GGUF, ~4.5 GB)
6. ¡Prueba Chat o Generate!

### Resumen de descargas

| Modelo | Origen | Descarga |
|--------|--------|----------|
| SDXL | Tu Drive (CivitAI merge) | ❌ No descarga nada |
| Qwen3.5-9B | HuggingFace (`HauhauCS/Qwen3.5-9B-Uncensored...`) | ✅ Una vez (se cachea) |

---
**ArtLab v2.0 — Backend Colab**
